# Phase 7 — Final comparison
Load both best-validation classifiers and evaluate them on the same ordered reserved test examples. Model A receives original images resized to 128×128. Model B receives on-the-fly SRGAN versions of those same sources; test images are never saved or used for training. The resulting table contains measured values only after this notebook runs.

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import MobileNet_V2_Weights

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, SRGANClassifierDataset, load_splits
from applied_ai_midterm.evaluation import build_model_comparison_table, evaluate_classifier
from applied_ai_midterm.models import Generator, build_mobilenet_v2_classifier
from applied_ai_midterm.training import load_best_classifier, load_best_generator, seed_everything, select_device
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
generator = Generator()
generator_checkpoint_path, _ = load_best_generator(
    generator, Path("checkpoints/srgan"), device
)
print(f"Using generator: {generator_checkpoint_path}")

In [ ]:
# Both loaders retain identical persisted test-row order and labels.
model_a_test_dataset = ImagePathDataset(
    splits.test, classifier_transform(training=False, image_size=config.high_resolution_size)
)
model_b_test_dataset = SRGANClassifierDataset(
    splits.test,
    generator,
    device,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
model_a_test_loader = DataLoader(
    model_a_test_dataset, batch_size=config.classifier_batch_size, shuffle=False, num_workers=config.num_workers
)
# Generator inference occurs inside this dataset, so workers remain zero for device safety.
model_b_test_loader = DataLoader(
    model_b_test_dataset, batch_size=config.classifier_batch_size, shuffle=False, num_workers=0
)

In [ ]:
# weights=None avoids unnecessary downloads because complete trained states are loaded next.
model_a = build_mobilenet_v2_classifier(weights=None, freeze_features=True)
model_b = build_mobilenet_v2_classifier(weights=None, freeze_features=True)
load_best_classifier(model_a, Path("checkpoints/model_a_best.pt"), device)
load_best_classifier(model_b, Path("checkpoints/model_b_best.pt"), device)
criterion = nn.BCEWithLogitsLoss()
_, model_a_metrics, model_a_labels, model_a_probabilities = evaluate_classifier(
    model_a, model_a_test_loader, criterion, device
)
_, model_b_metrics, model_b_labels, model_b_probabilities = evaluate_classifier(
    model_b, model_b_test_loader, criterion, device
)
assert (model_a_labels == model_b_labels).all()
assert len(model_a_probabilities) == len(model_b_probabilities) == len(splits.test)

In [ ]:
comparison_table = build_model_comparison_table(
    model_a_metrics,
    model_b_metrics,
    output_path=Path("artifacts/final_model_comparison.csv"),
)
display(comparison_table.style.format({column: "{:.4f}" for column in comparison_table.columns if column != "Model"}))